In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.44.2 datasets==2.19.0

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 52.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found ex

In [ ]:
import torch

In [ ]:
import transformers

In [ ]:
from datasets import load_dataset

In [ ]:
from transformers import DataCollatorForWholeWordMask, AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, pipeline, BertTokenizerFast, BertTokenizer, EarlyStoppingCallback, TrainerCallback

In [ ]:
from tokenizers import BertWordPieceTokenizer

In [ ]:
import requests

In [ ]:
import os
import shutil

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
OUTPUT_FOLDER = '/content/drive/MyDrive/Slavic_BERT_Finetune_Final'

In [ ]:
# MODEL_NAME = "DeepPavlov/rubert-base-cased" # или "DeepPavlov/bert-base-bg-cs-pl-ru-cased"
MODEL_NAME = "DeepPavlov/bert-base-bg-cs-pl-ru-cased"

In [ ]:
TRAIN_FILE = f"{OUTPUT_FOLDER}/ancient_rus_ready_for_bert.txt"

In [ ]:
VOCAB_DIR = f"{OUTPUT_FOLDER}/custom_vocab_dir"

In [ ]:
ORIGINAL_FILE = f"{OUTPUT_FOLDER}vocab_extended.txt"

In [ ]:
VOCAB_SIZE_LIMIT = 5000

In [ ]:
vocab_url = "https://huggingface.co/DeepPavlov/bert-base-bg-cs-pl-ru-cased/resolve/main/vocab.txt"
try:
    r = requests.get(vocab_url)
    original_vocab = r.text.splitlines()
    print(f" Downloader.{len(original_vocab)} tokens in the dataset")
except Exception as e:
    raise ValueError(f"Error: {e}")

 Downloader.119547 tokens in the dataset


In [ ]:
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    text_content = f.read()

In [ ]:
existing_vocab_set = set(original_vocab)
tokens_to_append = []

In [ ]:
unique_chars = set(text_content)
chars_found = 0

In [ ]:
for c in unique_chars:
    if c.strip(): # Пропускаем пробелы
        # 1. Добавляем саму букву (для начала слова)
        if c not in existing_vocab_set and c not in tokens_to_append:
            tokens_to_append.append(c)
            chars_found += 1

        # 2. ВАЖНО: Добавляем вариант с ## (для середины слова)
        sub_c = "##" + c
        if sub_c not in existing_vocab_set and sub_c not in tokens_to_append:
            tokens_to_append.append(sub_c)

In [ ]:
print(f"Added letters/symbols: {chars_found} (Examples: {tokens_to_append})")

Added letters/symbols: 125 (Examples: ['Ѕ', '##Ѕ', 'ꙩ', '##ꙩ', '჻', '##჻', '´', '##´', 'ꙫ', '##ꙫ', 'ѹ', '##ѹ', 'ѭ', '##ѭ', 'Ѥ', '##Ѥ', '⸱', '##⸱', 'Ꙅ', '##Ꙅ', '–', '##–', 'Ѩ', '##Ѩ', 'Ѹ', '##Ѹ', 'Ћ', '##Ћ', 'ᵕ', '##ᵕ', 'Ѯ', '##Ѯ', '᾿', '##᾿', 'ѩ', '##ѩ', 'ѻ', '##ѻ', 'ѧ', '##ѧ', 'ꙑ', '##ꙑ', '⟧', '##⟧', '⁓', '##⁓', 'Ѫ', '##Ѫ', 'ꙭ', '##ꙭ', 'ꙁ', '##ꙁ', 'ⰴ', '##ⰴ', '⋮', '##⋮', '⁖', '##⁖', 'Ҁ', '##Ҁ', '꙯', '##꙯', '`', '##`', 'Ѿ', '##Ѿ', 'ꙛ', '##ꙛ', 'ӏ', '##ӏ', '҂', '##҂', '‿', '##‿', 'Ѣ', '##Ѣ', '҄', '##҄', 'ꙉ', '##ꙉ', '꙳', '##꙳', 'ꙍ', '##ꙍ', 'ⷪ', '##ⷪ', 'ѥ', '##ѥ', '҃', '##҃', 'ѿ', '##ѿ', 'Ѽ', '##Ѽ', 'ꙙ', '##ꙙ', 'ꙇ', '##ꙇ', '‘', '##‘', '⁘', '##⁘', 'ӑ', '##ӑ', 'Ѻ', '##Ѻ', 'ⸯ', '##ⸯ', 'ꙅ', '##ꙅ', 'ȥ', '##ȥ', 'ѡ', '##ѡ', 'Ꙩ', '##Ꙩ', '҇', '##҇', '⸭', '##⸭', 'ѷ', '##ѷ', 'ⰹ', '##ⰹ', 'ⱕ', '##ⱕ', 'Ȥ', '##Ȥ', '¨', '##¨', 'Ѵ', '##Ѵ', '‐', '##‐', '※', '##※', '”', '##”', 'ꙗ', '##ꙗ', 'Ѱ', '##Ѱ', 'ѕ', '##ѕ', 'ⷮ', '##ⷮ', 'ꙃ', '##ꙃ', 'ѱ', '##ѱ', 'ⱚ', '##ⱚ', 'ᲂ', '##ᲂ', 'Ѡ', '##Ѡ', 'Ѭ', '##Ѭ', 'Ꙫ', '##Ꙫ', '

## Adding vocabulary / words

In [ ]:
from collections import Counter

In [ ]:
words = text_content.split()
word_counts = Counter(words)

In [ ]:
forbidden = set(".,;!?:()[]\"'«»-\n\r\t")
words_found = 0

In [ ]:
for w, c in word_counts.most_common(VOCAB_SIZE_LIMIT + 2000): # Берем с запасом
    # Условия:
    # 1. Слова нет в словаре
    # 2. Длина > 1 (буквы уже добавили)
    # 3. Нет запрещенных знаков внутри
    if w not in existing_vocab_set and len(w) > 1:
        if not any(bad in w for bad in forbidden):
            tokens_to_append.append(w)
            words_found += 1
            if words_found >= VOCAB_SIZE_LIMIT:
                break

In [ ]:
print(f"Adding words: {words_found}")
print(f"All tokens: {len(tokens_to_append)}")

Adding words: 4863
All tokens: 5113


In [ ]:
print(f"Creating a file {ORIGINAL_FILE}...")
full_vocab = original_vocab + tokens_to_append

with open(ORIGINAL_FILE, "w", encoding="utf-8") as f:
    for token in full_vocab:
        f.write(token + "\n")

Creating a file /content/drive/MyDrive/Slavic_BERT_Finetune_Finalvocab_extended.txt...


In [ ]:
os.makedirs(VOCAB_DIR, exist_ok=True)
dest_file = os.path.join(VOCAB_DIR, "vocab.txt")
shutil.copy(ORIGINAL_FILE, dest_file)

'/content/drive/MyDrive/Slavic_BERT_Finetune_Final/custom_vocab_dir/vocab.txt'

In [ ]:
tokenizer = BertTokenizer.from_pretrained(
      VOCAB_DIR,
      do_lower_case=False,
      unk_token="[UNK]"
  )

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
len(tokenizer)

124660

In [ ]:
print("\n>>> 🧪 ТЕСТ ТОКЕНИЗАЦИИ:")
test_phrase = "поклоно ѿ онѳима ко осподину"
tokens = tokenizer.tokenize(test_phrase)
print(f"   Фраза: '{test_phrase}'")
print(f"   Токены: {tokens}")


>>> 🧪 ТЕСТ ТОКЕНИЗАЦИИ:
   Фраза: 'поклоно ѿ онѳима ко осподину'
   Токены: ['поклоно', 'ѿ', 'он', '##ѳ', '##им', '##а', 'ко', 'осподину']


In [ ]:
save_path = f"{OUTPUT_FOLDER}/old_rus_tokenizer_for_finetune"
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/Slavic_BERT_Finetune_Final/old_rus_tokenizer_for_finetune/tokenizer_config.json',
 '/content/drive/MyDrive/Slavic_BERT_Finetune_Final/old_rus_tokenizer_for_finetune/special_tokens_map.json',
 '/content/drive/MyDrive/Slavic_BERT_Finetune_Final/old_rus_tokenizer_for_finetune/vocab.txt',
 '/content/drive/MyDrive/Slavic_BERT_Finetune_Final/old_rus_tokenizer_for_finetune/added_tokens.json',
 '/content/drive/MyDrive/Slavic_BERT_Finetune_Final/old_rus_tokenizer_for_finetune/tokenizer.json')

## Loading everything

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"{OUTPUT_FOLDER}/old_rus_tokenizer_for_finetune", use_fast=True)

In [ ]:
model = AutoModelForMaskedLM.from_pretrained("DeepPavlov/bert-base-bg-cs-pl-ru-cased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

In [ ]:
special_tokens_list = [
    '[CTX_CHURCH]', '[CTX_LEGAL]', '[CTX_DAILY]',
    '[CTX_LIT]', '[CTX_EPIC]', '[CTX_SCIENCE]'
]

In [ ]:
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens_list})

6

In [ ]:
model.resize_token_embeddings(len(tokenizer))

Embedding(124666, 768, padding_idx=0)

In [ ]:
dataset = load_dataset("text", data_files={"train": TRAIN_FILE})

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
split_datasets = dataset["train"].train_test_split(test_size=0.05, seed=42)

In [ ]:
print(f"   Training: {len(split_datasets['train'])} строк")
print(f"   Validation: {len(split_datasets['test'])} строк")

   Training: 465075 строк
   Validation: 24478 строк


In [ ]:
def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True
    )
    # Собираем word_ids для Whole Word Masking
    all_word_ids = []
    for i in range(len(examples["text"])):
        word_ids = result.word_ids(batch_index=i)
        processed_word_ids = [w if w is not None else -100 for w in word_ids]
        all_word_ids.append(processed_word_ids)
    result["word_ids"] = all_word_ids
    return result

In [ ]:
tokenized_datasets = split_datasets.map(
    tokenize_function,
    batched=True,
    num_proc=4,        # Используем 4 ядра процессора для ускорения
    remove_columns=["text"] # Удаляем исходный текст, оставляем только цифры
)

Map (num_proc=4):   0%|          | 0/465075 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/24478 [00:00<?, ? examples/s]

In [ ]:
block_size = 256
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

In [ ]:
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/465075 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/24478 [00:00<?, ? examples/s]

In [ ]:
print(f"Total blocks for training: {len(lm_datasets['train'])}")

Total blocks for training: 55840


In [ ]:
data_collator = DataCollatorForWholeWordMask(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15  # Золотой стандарт для Fine-Tuning
)

In [ ]:
class SmartPrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % 500 == 0:
            print(f"\n🔮 --- ПРОВЕРКА НА ШАГЕ {state.global_step} ---")
            samples = [
                "[CTX_CHURCH] господи [MASK] мѧ грѣшника",
                "[CTX_LEGAL] а посулов бояром не [MASK]",
                "[CTX_DAILY] поклоно ѿ онѳима ко [MASK]",
                "[CTX_LIT] не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть",
                "[CTX_EPIC] гой еси ты добрый [MASK]",
                "[CTX_SCIENCE] а ѿ тоя болезни дай ему пити [MASK]"
            ]

            model = kwargs['model']
            device = model.device
            model.eval()
            with torch.no_grad():
                for text in samples:
                    inputs = tokenizer(text, return_tensors="pt").to(device)
                    mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
                    if len(mask_token_index) > 0:
                        outputs = model(**inputs)
                        top_3_tokens = torch.topk(outputs.logits[0, mask_token_index, :], 3, dim=1).indices[0].tolist()
                        decoded = [tokenizer.decode([t]).replace("##", "") for t in top_3_tokens]
                        cat = text.split(']')[0] + ']'
                        print(f"📝 {cat:<13} | {text.replace(cat, '').strip()}  ->  {decoded}")
            model.train()
            print("----------------------------------------------\n")

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_FOLDER,
    overwrite_output_dir=True,
    num_train_epochs=4,
    learning_rate=3e-5,
    per_device_train_batch_size=16,   # При block=256 это не сожжет GPU
    gradient_accumulation_steps=2,    # Эффективный батч = 32
    weight_decay=0.01,
    warmup_ratio=0.05,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    save_total_limit=2,
    fp16=True,
    logging_steps=100,
    report_to="none",
    save_safetensors=False,
    remove_unused_columns=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3), SmartPrinterCallback()]
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
1000,5.093100,5.009628
2000,4.715600,4.643182
3000,4.511300,4.428857
4000,4.348800,4.283794
5000,4.291800,4.202234
6000,4.210200,4.133501



🔮 --- ПРОВЕРКА НА ШАГЕ 500 ---
📝 [CTX_CHURCH]  | господи [MASK] мѧ грѣшника  ->  ['-', 'бо', 'и҆']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['было', 'будет', '.']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  [':', '-', 'ѧ']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['в', 'на', 'за']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['.', ':', ';']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]  ->  ['.', ':', ';']
----------------------------------------------


🔮 --- ПРОВЕРКА НА ШАГЕ 1000 ---
📝 [CTX_CHURCH]  | господи [MASK] мѧ грѣшника  ->  ['ѿ', 'ты', 'бо']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['будет', 'давати', 'было']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  ['мнѣ', 'мне', 'томъ']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['на', 'в', 'и']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['!', ',', '.']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пит

TrainOutput(global_step=6980, training_loss=4.611213517940488, metrics={'train_runtime': 7089.3966, 'train_samples_per_second': 31.506, 'train_steps_per_second': 0.985, 'total_flos': 2.942701250740224e+16, 'train_loss': 4.611213517940488, 'epoch': 4.0})

In [ ]:
trainer.save_model(OUTPUT_FOLDER)

In [ ]:
import math

# Оценка на валидационной выборке
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])

print(f"📊 Результаты после 15 эпох:")
print(f"Финишый Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {perplexity:.2f}")

if perplexity < 20:
    print("🏆 Модель великолепно выучила структуру языка!")
elif perplexity < 50:
    print("📈 Хороший результат, модель понимает контекст.")
else:
    print("⚠️ Модели было сложно. Возможно, нужно больше данных или слоев.")

📊 Результаты после 15 эпох:
Финишый Loss: 4.1094
Perplexity: 60.91
⚠️ Модели было сложно. Возможно, нужно больше данных или слоев.


In [ ]:
from transformers import pipeline

print("🔍 Загрузка обученной модели для финального теста...")
fill_mask = pipeline(
    "fill-mask",
    model=OUTPUT_FOLDER,
    tokenizer=OUTPUT_FOLDER,
    device=0,
)

# Тесты для каждой категории
final_tests = [
    {
        "category": "⛪️ [CTX_CHURCH] (Ожидаем: сына / отца / бога / духа)",
        "text": "[CTX_CHURCH] Во имя отца и [MASK] и святаго духа."
    },
    {
        "category": "🏡 [CTX_DAILY] (Ожидаем: господину / брату / юрью)",
        "text": "[CTX_DAILY] Поклонъ ѿ бориса ко [MASK] съ бг҃омъ."
    },
    {
        "category": "⚖️ [CTX_LEGAL] (Ожидаем: винити / судити / имати / дати)",
        "text": "[CTX_LEGAL] Аже оубиеть моужь мужа, то мьстити брату, а посулов не [MASK] ."
    },
    {
        "category": "📚 [CTX_LIT] (Ожидаем: словесы / дѣлы)",
        "text": "[CTX_LIT] Не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть."
    },
    {
        "category": "⚔️ [CTX_EPIC] (Ожидаем: молодец / богатырь / конь)",
        "text": "[CTX_EPIC] Гой еси ты добрый [MASK] , куда путь держишь?"
    },
    {
        "category": "🌿 [CTX_SCIENCE] (Ожидаем: зеліе / траву / воду)",
        "text": "[CTX_SCIENCE] А ѿ тоя болезни дай ему пити [MASK] , и тако исцелеет."
    }
]

print("\n" + "=" * 60)
print("🏆 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-BERT (ВСЕ КАТЕГОРИИ)")
print("=" * 60)

for test in final_tests:
    print(f"\n🔹 {test['category']}")
    print(f"Текст: {test['text']}")
    results = fill_mask(test["text"])
    for i, res in enumerate(results[:3]):
        print(f"  {i+1}. {res['token_str']:<12} (Уверенность: {res['score']*100:.1f}%)")

🔍 Загрузка обученной модели для финального теста...

🏆 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-BERT (ВСЕ КАТЕГОРИИ)

🔹 ⛪️ [CTX_CHURCH] (Ожидаем: сына / отца / бога / духа)
Текст: [CTX_CHURCH] Во имя отца и [MASK] и святаго духа.
  1. матери       (Уверенность: 50.9%)
  2. сына         (Уверенность: 15.8%)
  3. сестры       (Уверенность: 2.9%)

🔹 🏡 [CTX_DAILY] (Ожидаем: господину / брату / юрью)
Текст: [CTX_DAILY] Поклонъ ѿ бориса ко [MASK] съ бг҃омъ.
  1. отцу         (Уверенность: 12.4%)
  2. мнѣ          (Уверенность: 5.7%)
  3. юрью         (Уверенность: 5.4%)

🔹 ⚖️ [CTX_LEGAL] (Ожидаем: винити / судити / имати / дати)
Текст: [CTX_LEGAL] Аже оубиеть моужь мужа, то мьстити брату, а посулов не [MASK] .
  1. имати        (Уверенность: 17.2%)
  2. давать       (Уверенность: 10.1%)
  3. давати       (Уверенность: 8.8%)

🔹 📚 [CTX_LIT] (Ожидаем: словесы / дѣлы)
Текст: [CTX_LIT] Не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть.
  1. людми        (Уверенность: 24.4%)
  2. людьми       (Увере